# 04 - Audit Kalender Ekonomi dan Fitur Makro (K3)

Notebook ini memverifikasi kesesuaian waktu rilis kalender ekonomi MT5 terhadap 10 jangkar waktu resmi (BLS & Federal Reserve lintas periode DST), lalu menyusun fitur makroekonomi tanpa kebocoran masa depan.

In [1]:
import pandas as pd

DATA_DIR = "../data/interim/phase4-p0"
PATH_KALENDER = "../data/raw/macro_calendar/mt5-economic-calendar-2020-2026.csv"

KODE_NUMERIK = ["US_CPI_MM", "US_CPI_YY", "US_CORE_CPI_MM", "US_CORE_CPI_YY", "US_NFP", "US_UNEMPLOYMENT"]
KODE_FLAG = ["US_RATE", "US_FOMC"]
BATAS_UMUR_JAM = 168
JENDELA_JAM = [1, 6, 24]

JANGKAR_RESMI = [
    ("US_NFP", "2020-01-10T13:30:00Z"),
    ("US_CPI_MM", "2020-01-14T13:30:00Z"),
    ("US_NFP", "2020-07-02T12:30:00Z"),
    ("US_CPI_MM", "2020-07-14T12:30:00Z"),
    ("US_NFP", "2020-11-06T13:30:00Z"),
    ("US_CPI_MM", "2020-11-12T13:30:00Z"),
    ("US_FOMC", "2024-01-31T19:00:00Z"),
    ("US_NFP", "2024-07-05T12:30:00Z"),
    ("US_CPI_MM", "2024-07-11T12:30:00Z"),
    ("US_FOMC", "2024-07-31T18:00:00Z"),
]

## 1. Baca Kalender & Cek 10 Jangkar Waktu Resmi (UTC)

Kolom `event_time_utc` dibuat sebagai kolom baru (tidak menimpa kolom waktu mentah `event_time_server`), lalu dicocokkan dengan 10 jadwal rilis resmi BLS dan Federal Reserve.

In [2]:
kal = pd.read_csv(PATH_KALENDER)
kal["event_time_utc"] = pd.to_datetime(kal["event_time_server"], format="%Y.%m.%dT%H:%M:%S", utc=True)

print("Jumlah baris kalender :", len(kal))
print("Duplikasi value_id    :", kal["value_id"].duplicated().sum())

jangkar = pd.DataFrame(JANGKAR_RESMI, columns=["event_code", "waktu_resmi_utc"])
jangkar["waktu_resmi_utc"] = pd.to_datetime(jangkar["waktu_resmi_utc"], utc=True)
jangkar["cocok"] = [
    ((kal["event_code"] == k) & (kal["event_time_utc"] == t)).sum() == 1
    for k, t in zip(jangkar["event_code"], jangkar["waktu_resmi_utc"])
]

STATUS_K3 = "LAYAK" if (not kal["value_id"].duplicated().any() and jangkar["cocok"].all()) else "NOT TESTED"
print("Status K3:", STATUS_K3)
jangkar

Jumlah baris kalender : 706
Duplikasi value_id    : 0
Status K3: LAYAK


,event_code,waktu_resmi_utc,cocok
0,US_NFP,2020-01-10 13:30:00+00:00,True
1,US_CPI_MM,2020-01-14 13:30:00+00:00,True
2,US_NFP,2020-07-02 12:30:00+00:00,True
3,US_CPI_MM,2020-07-14 12:30:00+00:00,True
4,US_NFP,2020-11-06 13:30:00+00:00,True
5,US_CPI_MM,2020-11-12 13:30:00+00:00,True
6,US_FOMC,2024-01-31 19:00:00+00:00,True
7,US_NFP,2024-07-05 12:30:00+00:00,True
8,US_CPI_MM,2024-07-11 12:30:00+00:00,True
9,US_FOMC,2024-07-31 18:00:00+00:00,True


## 2. Ringkasan Kalender per Kelompok Peristiwa

In [3]:
ringkas = kal.groupby("event_code").agg(
    jumlah_rilis=("value_id", "size"),
    actual_ada=("actual", lambda s: s.notna().sum()),
    forecast_ada=("forecast", lambda s: s.notna().sum()),
    pertama=("event_time_utc", "min"),
    terakhir=("event_time_utc", "max"),
)
ringkas

,jumlah_rilis,actual_ada,forecast_ada,pertama,terakhir
event_code,,,,,
US_CORE_CPI_MM,77,77,75,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_CORE_CPI_YY,78,78,77,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_CPI_MM,77,77,75,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_CPI_YY,78,78,77,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_FOMC,186,0,0,2020-01-03 19:00:00+00:00,2026-07-29 18:30:00+00:00
US_NFP,78,78,78,2020-01-10 13:30:00+00:00,2026-07-02 12:30:00+00:00
US_RATE,54,54,0,2020-01-29 19:00:00+00:00,2026-07-29 18:00:00+00:00
US_UNEMPLOYMENT,78,78,78,2020-01-10 13:30:00+00:00,2026-07-02 12:30:00+00:00


`US_RATE` dan `US_FOMC` tidak memiliki nilai *forecast*, sehingga tidak dihitung nilai kejutan (*surprise*) dan hanya menyumbang umur rilis serta penanda jendela waktu.

In [4]:
# Cek kelompok peristiwa yang memiliki lebih dari 1 baris pada menit yang sama
sama = kal[kal.duplicated(["event_code", "event_time_utc"], keep=False)]
print("Baris dengan waktu rilis sama:")
print(sama.groupby("event_code").size())

Baris dengan waktu rilis sama:
event_code
US_FOMC    50
dtype: int64


## 3. Bentuk Fitur Makroekonomi

Untuk setiap kelompok peristiwa dan setiap waktu keputusan $\tau_t$:
- Ambil rilis terakhir yang waktunya **lebih kecil secara ketat** dari $\tau_t$ (`allow_exact_matches=False`).
- `surprise = actual - forecast` (khusus 6 kelompok numerik).
- `age_hours` = umur rilis dalam jam, dibatasi maksimal 168 jam (7 hari); jika $\ge 168$ jam, `surprise` dikosongkan dan `expired = 1`.
- `within_1h`, `within_6h`, `within_24h` = penanda biner apakah rilis terjadi dalam rentang jam tersebut.

In [5]:
waktu = pd.read_parquet(f"{DATA_DIR}/02_label.parquet", columns=["time", "decision_time"])
kiri = waktu.reset_index()
makro = pd.DataFrame({"time": waktu["time"]})

for kode in KODE_NUMERIK + KODE_FLAG:
    prefix = f"macro_{kode.lower()}"

    ev = (
        kal[kal["event_code"] == kode]
        .sort_values(["event_time_utc", "value_id"])
        .drop_duplicates("event_time_utc", keep="last")
        .copy()
    )
    ev["surprise"] = pd.to_numeric(ev["actual"], errors="coerce") - pd.to_numeric(ev["forecast"], errors="coerce")

    gabung = pd.merge_asof(
        kiri,
        ev[["event_time_utc", "surprise"]],
        left_on="decision_time",
        right_on="event_time_utc",
        direction="backward",
        allow_exact_matches=False,
    ).set_index("index")

    umur = (gabung["decision_time"] - gabung["event_time_utc"]).dt.total_seconds() / 3600
    kedaluwarsa = umur >= BATAS_UMUR_JAM

    if kode in KODE_NUMERIK:
        makro[f"{prefix}_surprise"] = gabung["surprise"].where(~kedaluwarsa)
    makro[f"{prefix}_age_hours"] = umur.clip(upper=BATAS_UMUR_JAM)
    makro[f"{prefix}_expired"] = kedaluwarsa.astype(int)
    for jam in JENDELA_JAM:
        makro[f"{prefix}_within_{jam}h"] = ((umur > 0) & (umur <= jam)).astype(int)

kolom_makro = [c for c in makro.columns if c.startswith("macro_")]
print("Total fitur makro terbentuk:", len(kolom_makro))

Total fitur makro terbentuk: 46


### Contoh: Rilis NFP 5 Juli 2024 Pukul 12:30 UTC

Pada pukul 12:00 UTC rilis belum terlihat; pada pukul 13:00 UTC rilis baru terbaca dengan umur 0,5 jam.

In [6]:
contoh = makro.merge(waktu, on="time")
contoh = contoh[(contoh["decision_time"] >= "2024-07-05 11:00") & (contoh["decision_time"] <= "2024-07-05 16:00")]
contoh[["decision_time", "macro_us_nfp_surprise", "macro_us_nfp_age_hours", "macro_us_nfp_within_1h", "macro_us_nfp_within_6h"]]

,decision_time,macro_us_nfp_surprise,macro_us_nfp_age_hours,macro_us_nfp_within_1h,macro_us_nfp_within_6h
26650,2024-07-05 11:00:00+00:00,NaN,168.0,0,0
26651,2024-07-05 12:00:00+00:00,NaN,168.0,0,0
26652,2024-07-05 13:00:00+00:00,-81.0,0.5,1,1
26653,2024-07-05 14:00:00+00:00,-81.0,1.5,0,1
26654,2024-07-05 15:00:00+00:00,-81.0,2.5,0,1
26655,2024-07-05 16:00:00+00:00,-81.0,3.5,0,1


In [7]:
makro.to_parquet(f"{DATA_DIR}/04_fitur_makro.parquet", index=False)
with open(f"{DATA_DIR}/04_status_k3.txt", "w", encoding="utf-8") as f:
    f.write(STATUS_K3 + "\n")

print("Selesai! Tersimpan:", f"{DATA_DIR}/04_fitur_makro.parquet", makro.shape, "| Status K3:", STATUS_K3)

Selesai! Tersimpan: ../data/interim/phase4-p0/04_fitur_makro.parquet (38799, 47) | Status K3: LAYAK
